[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miguepoloc/toma-decisiones-mcda/blob/main/06_anp_cacao.ipynb)

# ANP, caso cacao (con interdependencia real)

Extiende el AHP de arriba: la accesibilidad de cada zona afecta qué tan confiable es cada criterio allí (pH/CE necesitan muestreo físico de suelo, difícil en zonas remotas; Temperatura/Humedad se aproximan con sensores remotos incluso en zonas remotas), y la accesibilidad misma depende del terreno de esa zona. Esa retroalimentación (Alternativas → Criterios) es lo que AHP no puede modelar y ANP sí. Contenido completo en la Sesión 6 (ANP) del curso.

In [1]:
!pip install -q pyDecision


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import numpy as np
from pyDecision.algorithm import ahp_method

criterios = ["Temperatura", "Humedad", "pH", "Conductividad"]
zonas = ["Bonda", "Guachaca", "San Pedro", "Palmor"]

## Paso 1, Bloque Criterios→Alternativas (ya conocido de AHP, S2)

In [3]:
CtoA = {
    "Bonda":     [0.4729, 0.4829, 0.2772, 0.3056],
    "Guachaca":  [0.2844, 0.2720, 0.1601, 0.0778],
    "San Pedro": [0.1699, 0.1570, 0.4673, 0.4918],
    "Palmor":    [0.0729, 0.0882, 0.0954, 0.1248],
}

## Paso 2, Bloque Alternativas→Criterios (NUEVO en ANP)

Dos perfiles de accesibilidad: Bonda y San Pedro (alta accesibilidad, pesos más balanceados), Guachaca y Palmor (baja accesibilidad, el peso se concentra en Temperatura/Humedad, obtenibles remotamente).

In [4]:
m_alta = np.array([
    [1,   1/2, 1,   2],
    [2,   1,   2,   3],
    [1,   1/2, 1,   2],
    [1/2, 1/3, 1/2, 1],
])
m_baja = np.array([
    [1,   1/2, 4,   4],
    [2,   1,   5,   5],
    [1/4, 1/5, 1,   1],
    [1/4, 1/5, 1,   1],
])
w_alta, cr_alta = ahp_method(m_alta, wd='m')
w_baja, cr_baja = ahp_method(m_baja, wd='m')
print("Perfil alta accesibilidad:", np.round(w_alta, 4), "CR=", round(cr_alta, 4))
print("Perfil baja accesibilidad:", np.round(w_baja, 4), "CR=", round(cr_baja, 4))

AtoC = {"Bonda": w_alta, "Guachaca": w_baja, "San Pedro": w_alta, "Palmor": w_baja}

Perfil alta accesibilidad: [0.2272 0.4231 0.2272 0.1225] CR= 0.0038
Perfil baja accesibilidad: [0.319  0.5017 0.0896 0.0896] CR= 0.0103


## Paso 3, Construir la supermatriz 8×8 (4 criterios + 4 alternativas)

In [5]:
n = 8
W = np.zeros((n, n))
# filas/columnas 0-3 = criterios, 4-7 = alternativas
for i, z in enumerate(zonas):
    W[4 + i, 0:4] = CtoA[z]        # Criterios -> Alternativas
    W[0:4, 4 + i] = AtoC[z]        # Alternativas -> Criterios

print("Suma de cada columna (debe ser 1.0):", np.round(W.sum(axis=0), 4))

Suma de cada columna (debe ser 1.0): [1.0001 1.0001 1.     1.     1.     1.     1.     1.    ]


## Paso 4, Supermatriz límite

La red es bipartita (2 clústeres, sin auto-influencia interna): las potencias pares e impares oscilan entre dos patrones, se promedian dos potencias consecutivas (Cesàro) para obtener la convergencia real.

In [6]:
W40 = np.linalg.matrix_power(W, 40)
W41 = np.linalg.matrix_power(W, 41)
W_limite = (W40 + W41) / 2

print("¿Todas las columnas de la supermatriz límite son iguales?",
      np.allclose(W_limite, W_limite[:, [0]] * np.ones((1, n)), atol=1e-3))

final = W_limite[:, 0]
final = final / final.sum()
for etiqueta, valor in zip(criterios + zonas, final):
    print(f"  {etiqueta:15s} {valor:.4f}")

¿Todas las columnas de la supermatriz límite son iguales? True
  Temperatura     0.1284
  Humedad         0.2242
  pH              0.0914
  Conductividad   0.0560
  Bonda           0.2114
  Guachaca        0.1165
  San Pedro       0.1272
  Palmor          0.0448


## Paso 5, Ranking final (renormalizado solo entre zonas)

In [7]:
prioridad_zonas = final[4:8]
prioridad_zonas = prioridad_zonas / prioridad_zonas.sum()
print("Ranking final ANP:")
for z, p in sorted(zip(zonas, prioridad_zonas), key=lambda x: -x[1]):
    print(f"  {z:12s} {p:.4f}")

Ranking final ANP:
  Bonda        0.4229
  San Pedro    0.2545
  Guachaca     0.2330
  Palmor       0.0897


**Resultado esperado** (coincide con lo publicado en la Sesión 6 del curso): 1º Bonda (0.4230) · 2º San Pedro (0.2542) · 3º Guachaca (0.2331) · 4º Palmor (0.0897), idéntico a AHP, VIKOR y PROMETHEE, pero los pesos de criterios SÍ cambian frente a AHP puro (Humedad 52.2%→45.0%, Temperatura 20.0%→25.6%) porque ahora dependen también de la accesibilidad de cada zona.